In [1]:
import ast
from dimension import Dim, KnownDim, BinaryDim, DimDecl, SymDim
from tensor_decl import TensorDecl
from custom_types import Type, MatrixType  # !!!!! CUSTOM TYPES
from torch_parser import TorchOpParser, TensorLiteralExpr

# could just return a MatrixShape? or even a tuple
class TensorLiteralExpr:
    def __init__(self, shape):
        self.shape: tuple[int, int] = shape

class RandnExpr:
    def __init__(self, shape):
        self.shape = shape

class MatMulExpr:
    def __init__(self, left, right):
        self.left = left
        self.right = right


class Env:  # store str: Type -- Type is custom defined in this project
    def __init__(self):
        self.env = []
        self.push()

    def push(self) -> None:
        self.env.append({})

    def pop(self) -> None:
        self.env.pop()

    def declare(self, name: str, type: Type) -> None:
        self.env[-1][name] = type

    def lookup(self, name: str) -> Type:
        for frame in reversed(self.env):
            if name in frame:
                return frame[name]
        raise KeyError(name)


class SemanticBuilder(ast.NodeVisitor):  # for now only support torch.tensor and int, extend to include other things
    def __init__(self, env: Env):
        self.ir = []  # just a list of nodes (declarations for now, can extend later)
        self.env = env  # !! env is passed to EVERY visitor, this, constraint, IR, everyone needs the SAME one
        self.torchParser = TorchOpParser()

    def visit_FunctionDef(self, node: ast.FunctionDef):  # ensure new scope !!!!!!!!
        self.env.push()
        self.generic_visit(node)  # recursive, comes back to parse inside of function params
        self.env.pop()

    def visit_AnnAssign(self, node: ast.AnnAssign) -> None:
        annotation = node.annotation
        identifier = node.target.id
        value = node.value

        # only parse that which is claimed to be integer, ignore the rest, they can't represent dimensions
        if isinstance(annotation, ast.Name) and (annotation.id == "int"):  # we are in a 'x: int = 4' type of statement
            value = Dim.toDim(node.value)
            self.ir.append(DimDecl(identifier, value))
            self.env.declare(identifier, value)  # is this valid? Dim not really a type
            return

        if (  # will break if using some alias, to fix
        isinstance(annotation, ast.Subscript)
        and isinstance(annotation.value, ast.Attribute)
        and annotation.value.attr == "Tensor"
        and isinstance(annotation.value.value, ast.Name)
        and annotation.value.value.id == "torch"
        ):  # to do fix this to rely on Dim rather than int and str
            dims = [Dim.toDim(i) for i in annotation.slice.elts]  # List(elts...) -> [Dim, Dim]
            tensorValue = self.torchParser.parse(value)
            self.ir.append(TensorDecl(identifier, (dims[0], dims[1]), tensorValue))  # tensorValue encodes information relevant for constraints, shapes
            self.env.declare(identifier, MatrixType(dims[0], dims[1]))
            return


In [ ]:
from z3 import Int, IntVal, AstRef, sat, Solver
from rules import Rules
from custom_literals import *


class ConstraintSolver:
    def __init__(self, env, report_errors=True, crash_if_error=False):
        self.solver = Solver()
        self.errors = []  # append errors as we go, inform the user and crash at the end only if provided crash=true arg?
        self.rules = Rules()
        self.env = env  # shared context

    def to_z3(self, expr: Dim) -> AstRef:  # fold simple constants
        if isinstance(expr, KnownDim):
            return IntVal(expr.value)
        elif isinstance(expr, SymDim):  # Dim should actually be the interface, SymbolicDim should inherit, else this is too fragile
            return Int(expr.name)
        elif isinstance(expr, BinaryDim):
            lhs = self.to_z3(expr.left)
            rhs = self.to_z3(expr.right)
            op = expr.operator
            if op == "+":
                return lhs + rhs
            elif op == "-":
                return lhs - rhs
            elif op == "*":
                return lhs * rhs
            elif op == "/":
                return lhs / rhs
        raise TypeError(f"Unsupported expr: {expr}")
    
    def get_var(self, name):
        return Int(name)

    def solve(self, semantic_ir) -> bool:
        for node in semantic_ir:

            if isinstance(node, DimDecl) and node.value is not None:
                z3_expr = self.to_z3(node.value)
                var = self.get_var(node.name)
                self.solver.add(var == z3_expr)
                self.solver.add(var > 0)

            # type hint shapes and runtime shapes must align via z3
            if isinstance(node, TensorDecl) and isinstance(node.value, TensorLiteralExpr):  # declared tensor with elements, ex: torch.tensor([[...]])
                shape_val = node.value.shape
                rows_val = self.to_z3(KnownDim(shape_val[0]))  # operate on the assumption that shape must contain int, this always holds true
                cols_val = self.to_z3(KnownDim(shape_val[1]))
                rows_hint = self.to_z3(node.shape[0])
                cols_hint = self.to_z3(node.shape[1])
                self.solver.add(rows_hint == rows_val)
                self.solver.add(cols_hint == cols_val)

            if isinstance(node, TensorDecl) and isinstance(node.value, MatMulExpr):
                ltype = self.env.lookup(node.value.left)
                rtype = self.env.lookup(node.value.right)
                self.solver.add(  # will dispatch this back to Rules eventually, fow now it's fine, this works
                    self.to_z3(ltype.cols) == self.to_z3(rtype.rows)
                )
                self.solver.add(
                    self.to_z3(node.shape[0]) == self.to_z3(ltype.rows)
                )
                self.solver.add(
                    self.to_z3(node.shape[1]) == self.to_z3(rtype.cols)
                )

            if isinstance(node, TensorDecl) and isinstance(node.value, AddExpr):
                ltype = self.env.lookup(node.value.left)
                rtype = self.env.lookup(node.value.right)
                # self.rules.apply("add", ltype, rtype, )
                self.solver.add(
                    self.to_z3(ltype.rows) == self.to_z3(rtype.rows)
                )
                self.solver.add(
                    self.to_z3(ltype.cols) == self.to_z3(rtype.cols)
                )
                self.solver.add(
                    self.to_z3(node.shape[0]) == self.to_z3(ltype.rows)
                )
                self.solver.add(
                    self.to_z3(node.shape[1]) == self.to_z3(ltype.cols)
                )

            if isinstance(node, TensorDecl) and isinstance(node.value, RandnExpr):
                shape_val = node.value.shape
                rows_val = self.to_z3(shape_val[0])
                cols_val = self.to_z3(shape_val[1])
                rows_hint = self.to_z3(node.shape[0])
                cols_hint = self.to_z3(node.shape[1])
                self.solver.add(rows_hint == rows_val)
                self.solver.add(cols_hint == cols_val)

        return self.solver.check() == sat

In [3]:
def run(code: str):
    tree = ast.parse(code)
    env = Env()  # same context to be used in all visitors

    builder = SemanticBuilder(env)
    builder.visit(tree)

    c_solver = ConstraintSolver(env, True, False)
    print(c_solver.solve(builder.ir))

    return builder

In [4]:
def dump(builder: SemanticBuilder):
    for item in builder.ir:
        print(item)

In [7]:
code = """
n: int = 1
m: int = 3
k: int = 1

A: torch.Tensor[n, m] = torch.tensor([[1, 2, 3]])
B: torch.Tensor[m, k] = torch.tensor([[1], [2], [3]])

C: torch.Tensor[n, k] = torch.matmul(A, B)
D: torch.Tensor[3, 3] = torch.randn(3, 3)
E: torch.Tensor[3, 3] = torch.add(D, D)
"""

builder = run(code)
# dump(builder)

True


In [6]:
from __future__ import annotations
import torch


n: int = 1
m: int = 3
k: int = 1
p: int = 9

A: torch.Tensor[n, m] = torch.tensor([[1, 2, 3]])
B: torch.Tensor[m, k] = torch.tensor([[1], [2], [3]])

C: torch.Tensor[n, p] = torch.matmul(A, B)

print(C)

tensor([[14]])
